<a href="https://colab.research.google.com/github/Tomoki4e7e/DLBasic2026_Final_Assignment_VQA/blob/softlabel/DL_Basic_2026_Spring_Competition_VQA_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning 基礎講座　最終課題: VQA

## 概要
画像と質問から，回答を予測するタスクです．
- サンプル数: 訓練 19,873 サンプル，テスト 4,969 サンプル
- 入力: 画像データ（RGB，サイズは画像によって異なります），質問文（系列長はサンプルごとに異なります）
- 出力: 回答文（系列長はサンプルごとに異なります）
- 評価指標: VQA での評価指標（[こちら
](https://visualqa.org/evaluation.html)を参照）を利用しています．

### データセット ([VizWiz 2023 edition](https://www.kaggle.com/datasets/nqa112/vizwiz-2023-edition)) の詳細
- 24,842 枚の画像データセットと，各画像に対する 1 つの質問文と 10 人の回答者による回答文から構成されます．
  - 10 人の回答は全て同じとは限りません．
- 24.842 サンプルのうち，80 % (19.873) が訓練データ (train)，20 % (4969) がテストデータ (val) として与えられます．
  - テストデータに対する回答文を正解ラベルとし，配布していません．
  - データ提供元とは異なるデータ分割になっています．

### タスクの詳細
- 本コンペティションでは，与えられた画像と質問文に対して，適切な回答文を出力するモデルを作成していただきます．
- 評価は [VQA](https://visualqa.org/index.html) (Visual Question Answering) に基づいて，以下の式で計算されます．

$$\text{Acc}(ans) = \text{min}(\frac{humans \; that \; said \; ans}{3}, 1)$$

- 1 つのデータに対し， 10 人の回答のうち 9 人の回答を選択し上記の式で性能評価した， 10 パターンの Acc の平均をそのデータに対する Acc とします．
- 予測結果と正解ラベルを比較する前に，回答を lowercase にする，冠詞は削除するなどの前処理を行っています（[詳細](https://visualqa.org/evaluation.html)）．

## 考えられる工夫の例
- 事前学習モデルの fine-tuning
    - 画像特徴量，言語特徴量を取得するときに，事前学習モデルを fine-tuning することで性能向上が見込めます（今回のタスクと大きく異なるデータセットでの事前学習では効果が小さい可能性がありますので注意しましょう）．
- 質問文の表現
    - ベースラインでは，質問文をモデルに入力する際に，one-hot ベクトルにしています．これを tokenizer 等を利用して分散表現にすることで，モデル学習しやすくなります．
- ソフトラベルの利用
    - ベースラインでは 10 人の回答の中で最も多かった回答を正解ラベルとして訓練しています．この点を各回答の頻度に合わせてソフトラベルを利用することで，より多くの情報を利用して学習が可能になります．
- 画像の前処理
    - 画像の前処理には形状を同じにする Resize のみを利用しています．「畳み込みニューラルネットワーク」，「深層学習と画像認識」等で紹介されていたデータ拡張を追加することで，汎化性能の向上が見込めます．

## 修了要件を満たす条件
- ベースラインでは，omnicampus 上での性能評価において， 49.9% となります．したがって，ベースラインを超える 49.9% を超えた提出のみ，修了要件として認めます．
- ベースラインから改善を加えることで， 60% に性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

### データの準備
データをダウンロードした際に，google drive したため，利用するために google drive をマウントする必要があります．また， drive 上で展開することができないため，/content ディレクトリ下にコピーし "data.zip" を展開します．  
google drive 上に "data.zip" が配置されていない場合は実行できません．google drive 上に "data.zip" (**12GB**) を配置することが可能であれば，"data_download.ipynb" を先に実行してください．難しい場合は，omnicampus 演習環境を利用してください．．



In [41]:
# omnicampus 上では 4 セル目まで実行不要
# ドライブのマウント
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
import os

base_dir = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment"
zip_path = os.path.join(base_dir, "data.zip")
local_zip_path = "/content/data.zip"
local_data_dir = "/content/data"

# 既存のシンボリックリンクがあれば削除
if os.path.islink(local_data_dir):
    os.unlink(local_data_dir)
    print("古いシンボリックリンクを削除しました。")

if os.path.exists(local_data_dir):
    print("既に解凍済みのローカルデータが存在します。スキップします。")
else:
    print("Google Driveからデータをローカル環境にコピーします...")
    !cp "{zip_path}" "{local_zip_path}"
    print("コピー完了。解凍を開始します...")
    !unzip -q "{local_zip_path}" -d "/content/"
    print("ローカル環境へのデータ展開が完了しました。")


既に解凍済みのローカルデータが存在します。スキップします。


In [43]:
# カレントディレクトリ下のファイル群を確認
# data が水色（シンボリックリンク）で表示されていれば問題ありません
%ls -l


total 11682740
drwxr-xr-x 4 root root        4096 Jun  7 07:25 data/
-rw------- 1 root root 11963107886 Jun 21 11:37 data.zip
drwx------ 5 root root        4096 Jun 21 11:33 drive/
drwxr-xr-x 1 root root        4096 Jun  4 13:39 sample_data/


In [44]:
# データダウンロード用の notebook にてgoogle drive への保存後，
# 反映に時間がかかる可能性がありますので，google drive のマウント後，
# data.zip がディレクトリ内にあることを確認してから実行してください．
# data.zip を /content 下にコピーする
# !cp "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/data.zip" "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment"

In [45]:
# カレントディレクトリ下のファイル群を確認
# data.zip が表示されれば問題ないです
# %ls

In [46]:
# データを解凍する
# !unzip data.zip

### 1. import library

In [47]:
!pip install transformers

In [48]:
import re
import random
import time
from statistics import mode

from PIL import Image
import numpy as np
import pandas
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from collections import Counter
from transformers import AutoTokenizer, AutoModel

### 2. utils

In [49]:
def set_seed(seed):
    """
    シードを固定する．

    Parameters
    ----------
    seed : int
        乱数生成に用いるシード値．
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [50]:
def process_text(text):
    """
    入力文と回答のフォーマットを統一するための関数．

    Parameters
    ----------
    text : str
        入力文，もしくは回答．
    """
    # lowercase
    text = text.lower()

    # 数詞を数字に変換
    num_word_to_digit = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'ten': '10'
    }
    for word, digit in num_word_to_digit.items():
        text = text.replace(word, digit)

    # 小数点のピリオドを削除
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)

    # 冠詞の削除
    text = re.sub(r'\b(a|an|the)\b', '', text)

    # 短縮形のカンマの追加
    contractions = {
        "dont": "don't", "isnt": "isn't", "arent": "aren't", "wont": "won't",
        "cant": "can't", "wouldnt": "wouldn't", "couldnt": "couldn't"
    }
    for contraction, correct in contractions.items():
        text = text.replace(contraction, correct)

    # 句読点をスペースに変換
    text = re.sub(r"[^\w\s':]", ' ', text)

    # 句読点をスペースに変換
    text = re.sub(r'\s+,', ',', text)

    # 連続するスペースを1つに変換
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [51]:
class VQADataset(torch.utils.data.Dataset):
    """
    VQA データセットを扱うためのクラス．
    """
    def __init__(self, df_path, image_dir, transform=None, answer=True):
        self.transform = transform  # 画像の前処理
        self.image_dir = image_dir  # 画像ファイルのディレクトリ
        self.df = pandas.read_json(df_path)  # 画像ファイルのパス，question, answerを持つDataFrame
        self.answer = answer

        # --- 変更点: AutoTokenizerの利用 ---
        self.text_model_name = 'roberta-base' # 'microsoft/deberta-v3-base' などに変更可能
        self.tokenizer = AutoTokenizer.from_pretrained(self.text_model_name)

        self.answer2idx = {}
        self.idx2answer = {}

        if self.answer:
            # 回答に含まれる文章を辞書に追加
            for answers in self.df["answers"]:
                for answer in answers:
                    word = answer["answer"]
                    word = process_text(word)
                    if word not in self.answer2idx:
                        self.answer2idx[word] = len(self.answer2idx)
            self.idx2answer = {v: k for k, v in self.answer2idx.items()}  # 逆変換用の辞書(answer)

    def update_dict(self, dataset):
        """
        検証用データ，テストデータの辞書を訓練データの辞書に更新する．
        """
        self.answer2idx = dataset.answer2idx
        self.idx2answer = dataset.idx2answer

    def __getitem__(self, idx):
        """
        対応するidxのデータ（画像，質問，回答）を取得．
        """
        image = Image.open(f"{self.image_dir}/{self.df['image'][idx]}")
        image = self.transform(image)

        question_text = self.df["question"][idx]

        encoded = self.tokenizer(
            question_text,
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors='pt'
        )

        inputs_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        if self.answer:
            answers = [self.answer2idx[process_text(answer["answer"])] for answer in self.df["answers"][idx]]

            soft_label = torch.zeros(len(self.answer2idx))
            answer_counts = Counter(answers)

            for ans_id, count in answer_counts.items():
                soft_label[ans_id] = min(count / 3.0, 1.0)

            return image, inputs_ids, attention_mask, torch.Tensor(answers), soft_label
        else:
            return image, inputs_ids, attention_mask

    def __len__(self):
        return len(self.df)

In [52]:
def VQA_criterion(batch_pred, batch_answers):
    """
    VQA タスクに用いられる評価関数．
    """
    total_acc = 0.

    for pred, answers in zip(batch_pred, batch_answers):
        acc = 0.
        for i in range(len(answers)):
            num_match = 0
            for j in range(len(answers)):
                if i == j:
                    continue
                if pred == answers[j]:
                    num_match += 1
            acc += min(num_match / 3, 1)
        total_acc += acc / 10

    return total_acc / len(batch_pred)

### 3. model

In [53]:
class BasicBlock(nn.Module):
    """
    ResNet の basic block
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out


class BottleneckBlock(nn.Module):
    """
    ResNet の bottleneck block
    """
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, stride=1)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out


class ResNet(nn.Module):
    """
    ResNet の実装
    """
    def __init__(self, block, layers):
        """
        コンストラクタ．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        layers: list
            各ブロックの層数
        """
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, layers[0], 64)
        self.layer2 = self._make_layer(block, layers[1], 128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], 256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], 512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, 512)

    def _make_layer(self, block, blocks, out_channels, stride=1):
        """
        同じ構成を繰り返す部分を生成する．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        blocks: int
            層数
        out_channels: int
            出力のチャネル数
        stride: int
            ストライド

        Returns
        -------
        layers: torch.nn.ModuleList
            生成した層
        """
        layers = []
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            入力データ

        Returns
        -------
        x: torch.Tensor
            ResNet によって生成される特徴量
        """
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [54]:
def ResNet18():
    """
    ResNet18 を生成する関数．
    """
    return ResNet(BasicBlock, [2, 2, 2, 2])


def ResNet50():
    """
    ResNet50 を生成する関数．
    """
    return ResNet(BottleneckBlock, [3, 4, 6, 3])

In [55]:
class VQAModel(nn.Module):
    """
    VQA タスクを解くためのモデル例．
    """
    def __init__(self, n_answer: int):
        """
        コンストラクタ．

        Parameters
        ----------
        n_answer: int
            出力のクラス数
        """
        super().__init__()
        # 1. 事前学習済みVision Transformer (ViT-B/16) のロード
        self.vit = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
        # 分類ヘッドを無効化して特徴量抽出器として利用 (出力次元: 768)
        self.vit.heads = nn.Identity()

        # --- 変更点: AutoModelの利用 ---
        text_model_name = 'roberta-base' # 'microsoft/deberta-v3-base' などに変更可能
        self.text_encoder = AutoModel.from_pretrained(text_model_name)

        # --- 変更点: Cross-Attention機構の導入 ---
        # 次元数768、ヘッド数8のMultiheadAttention
        self.cross_attn = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        # Attentionを通った特徴量(768)と、元の画像特徴量(768)を結合して出力層へ
        self.fc = nn.Sequential(
            nn.Linear(768 * 2, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(512, n_answer)
        )

    def forward(self, image, input_ids, attention_mask):
        # 画像の特徴量を取得: [batch_size, 768]
        image_feature = self.vit(image)

        # テキストの系列全体の特徴量を取得: [batch_size, seq_len, 768]
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_seq = text_outputs.last_hidden_state

        # --- Cross-Attentionの適用 ---
        # 画像特徴量をQueryとして [batch_size, 1, 768] の形状にする
        query = image_feature.unsqueeze(1)
        # テキスト特徴量をKey, Valueとする
        key = text_seq
        value = text_seq

        # BERTのpadding部分をAttentionから除外するマスクを作成
        # (attention_maskは 1が有効トークン、0がpadding。key_padding_maskは Trueの場所を無視する)
        key_padding_mask = (attention_mask == 0)

        # Attentionの計算: attn_outputは [batch_size, 1, 768]
        attn_output, _ = self.cross_attn(query, key, value, key_padding_mask=key_padding_mask)
        attn_output = attn_output.squeeze(1) # [batch_size, 768] に戻す

        # 元の画像特徴量とAttentionの出力を結合して予測
        x = torch.cat([image_feature, attn_output], dim=1)
        x = self.fc(x)

        return x

### 4. train

In [56]:
def train(model, dataloader, optimizer, criterion, device):
    """
    学習用の関数．
    """
    model.train()

    total_loss = 0
    total_acc = 0
    simple_acc = 0

    start = time.time()
    for image, input_id, attention_mask, answers, soft_label in dataloader:
        image, input_id, attention_mask, answers, soft_label = \
            image.to(device), input_id.to(device), attention_mask.to(device), answers.to(device), soft_label.to(device)

        pred = model(image, input_id, attention_mask)
        loss = criterion(pred, soft_label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
        simple_acc += (pred.argmax(1) == soft_label.argmax(1)).float().mean().item()  # simple accuracy

    return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start


def eval(model, dataloader, optimizer, criterion, device):
    """
    評価用の関数．
    """
    model.eval()

    total_loss = 0
    total_acc = 0
    simple_acc = 0

    start = time.time()
    for image, input_id, attention_mask, answers, soft_label in dataloader:
        image, input_id, attention_mask, answers, soft_label = \
            image.to(device), input_id.to(device), attention_mask.to(device), answers.to(device), soft_label.to(device)

        pred = model(image, input_id, attention_mask)
        loss = criterion(pred, soft_label)

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
        simple_acc += (pred.argmax(1) == soft_label.argmax(1)).float().mean().item()  # simple accuracy

    return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start

### 5. make submission file

In [57]:
# deviceの設定
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. VQA向けの安全なData Augmentation
# 反転や大きな回転は避け、軽微なクロップと色調変化に留める
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # ズームインによる位置ズレ
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.0), # 軽微な色変化
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# train_datasetにtransform_trainを適用
train_dataset = VQADataset(df_path="./data/train.json", image_dir="./data/train", transform=transform_train)
test_dataset = VQADataset(df_path="./data/valid.json", image_dir="./data/valid", transform=transform, answer=False)
test_dataset.update_dict(train_dataset)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

model = VQAModel(n_answer=len(train_dataset.answer2idx)).to(device)

# optimizer / criterion
num_epoch = 4
criterion = nn.BCEWithLogitsLoss()

# 3. 層ごとの学習率（Parameter Groups）の設定
# Cross-Attention層を最適化対象に含める
optimizer = torch.optim.Adam([
    {'params': model.vit.parameters(), 'lr': 1e-5},
    {'params': model.text_encoder.parameters(), 'lr': 1e-5},
    {'params': model.cross_attn.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-4}
], weight_decay=1e-5)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [58]:
# num_epoch = 4
for epoch in range(num_epoch):
    train_loss, train_acc, train_simple_acc, train_time = train(model, train_loader, optimizer, criterion, device)
    print(f"【{epoch + 1}/{num_epoch}】\n"
            f"train time: {train_time:.2f} [s]\n"
            f"train loss: {train_loss:.8f}\n"
            f"train acc: {train_acc:.4f}\n"
            f"train simple acc: {train_simple_acc:.4f}")


【1/4】
train time: 918.12 [s]
train loss: 0.02600050
train acc: 0.1479
train simple acc: 0.1351
【2/4】
train time: 919.87 [s]
train loss: 0.00073257
train acc: 0.2794
train simple acc: 0.2547
【3/4】
train time: 918.30 [s]
train loss: 0.00064426
train acc: 0.3418
train simple acc: 0.3120
【4/4】
train time: 918.34 [s]
train loss: 0.00059324
train acc: 0.4038
train simple acc: 0.3687


In [59]:
# make submission file
model.eval()
submission = []
for image, input_ids, attention_mask in test_loader:
    image, input_ids, attention_mask = image.to(device), input_ids.to(device), attention_mask.to(device)
    pred = model(image, input_ids, attention_mask)
    pred = pred.argmax(1).cpu().item()
    submission.append(pred)

submission = [train_dataset.idx2answer[id] for id in submission]
submission = np.array(submission)
torch.save(model.state_dict(), "model.pt")
np.save("submission.npy", submission)

KeyboardInterrupt: 

## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (VQA)」から提出してください．

- `submission.npy`
- `model.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [ ]:
from zipfile import ZipFile

zip_save_path = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/submission.zip"
model_path = "model.pt"
notebook_path = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/DL_Basic_2026_Spring_Competition_VQA_baseline.ipynb"

with ZipFile(zip_save_path, "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path, arcname="DL_Basic_2026_Spring_Competition_VQA_baseline.ipynb")

In [ ]:
# === 2. Flash Log (2.5 Flash + CSV) ===
!pip install -q -U google-genai gradio

import csv
import datetime
import os
import gradio as gr
from google import genai
from google.colab import userdata

# APIキーの準備
api_key = userdata.get('Colab_chat_logger')
if not api_key: raise RuntimeError("APIキーを設定してください")
client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-2.5-flash"
LOG_FILE = "chat_log.csv"

# --- 追加機能：ログ保存関数 ---
def save_log(user_text, ai_text):
    # ファイルがまだなければ、ヘッダー（項目名）を書き込む準備をする
    file_exists = os.path.isfile(LOG_FILE)

    # "a" (append) モードでファイルを開く
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # 初回だけヘッダーを書き込む
        if not file_exists:
            writer.writerow(["Timestamp", "User", "AI"])

        # 日時・質問・回答を1行にして書き込む
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        writer.writerow([timestamp, user_text, ai_text])

def chat(message, history):
    chat_history = []
    if history is None: history = []

    # 履歴の変換（前回と同じ）
    for entry in history:
        if isinstance(entry, (list, tuple)):
            u, a = entry
            if u: chat_history.append(genai.types.Content(role="user", parts=[genai.types.Part.from_text(text=str(u))]))
            if a: chat_history.append(genai.types.Content(role="model", parts=[genai.types.Part.from_text(text=str(a))]))
        elif isinstance(entry, dict):
            role = "user" if entry["role"] == "user" else "model"
            if entry["content"]:
                chat_history.append(genai.types.Content(role=role, parts=[genai.types.Part.from_text(text=str(entry["content"]))]))

    chat_history.append(genai.types.Content(role="user", parts=[genai.types.Part.from_text(text=message)]))

    try:
        # Geminiから回答を取得
        response = client.models.generate_content(model=MODEL_NAME, contents=chat_history)
        ai_response = response.text

        # ★ここでログ保存を実行！
        save_log(message, ai_response)

        return ai_response
    except Exception as e:
        return f"エラー: {e}"

# UIの起動
gr.ChatInterface(fn=chat, title="Gemini 2.5 Flash Log").launch(share=True)

